In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from anthropic.types import Message

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
# --- Helpers ---
def add_user_message(messages, message):
    messages.append({"role": "user", "content": message.content if isinstance(message, Message) else message})

def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message.content if isinstance(message, Message) else message})

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {"model": model, "max_tokens": 4096, "messages": messages, "temperature": temperature, "stop_sequences": stop_sequences}
    if tools: params["tools"] = tools
    if system: params["system"] = system
    return client.messages.create(**params)

def text_from_message(message):
    return "\n".join(block.text for block in message.content if hasattr(block, "text"))

In [ ]:
# --- Web Search Schema ---
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
}

In [ ]:
# Test 1: General web search
messages = []
add_user_message(messages, "What are the latest developments in quantum computing in 2026?")
response = chat(messages, tools=[web_search_schema])

# Show all block types
for block in response.content:
    if block.type == "text":
        print(f"[text] {block.text[:100]}...")
    elif block.type == "server_tool_use":
        print(f"[search] query: {block.input.get('query')}")
    elif block.type == "web_search_tool_result":
        for r in block.content:
            if r.type == "web_search_result":
                print(f"  [result] {r.title} — {r.url}")

In [ ]:
# Test 2: Domain-restricted search (nih.gov only)
web_search_nih = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

messages = []
add_user_message(messages, "What's the best exercise for gaining leg muscle?")
response = chat(messages, tools=[web_search_nih])

for block in response.content:
    if block.type == "server_tool_use":
        print(f"Query: {block.input.get('query')}")
    elif block.type == "web_search_tool_result":
        for r in block.content:
            if r.type == "web_search_result":
                print(f"  {r.title} — {r.url}")

print("\n" + text_from_message(response))